# NatureCubePy Data Retrieval Tutorial

This notebook demonstrates NatureCubePy's data retrieval functions for accessing biodiversity observation data across multiple measurement types: camera traps (image/video), bioacoustics (audio), and environmental DNA (eDNA).

## Data Structure Overview

NatureCube projects consist of:
- **Stations**: Physical or logical measurement deployments (camera traps, audio recorders, eDNA collection sites)
- **Media Assets**: Files recorded at stations (images, video segments, audio clips)
- **Media Segments**: Time-bounded regions within an asset (e.g., video clip 0:30-0:45, species present in audio segment)
- **Labels**: Species identifications or other annotations on segments

All observations are geotagged with station coordinates and include measurement type and device metadata.

## API Tutorial Flow

1. **Stations**: Retrieve geolocations and metadata for all deployment sites
2. **Labels**: Access project-specific species reference data
3. **Media**: Query raw assets and their temporal segments
4. **Species Observations**: Retrieve merged records with species IDs, locations, and measurement context

Prereqs: a valid Okala API key (follow [01_authentication.ipynb](./01_authentication.ipynb) to set up) and NatureCubePy installed. 

In [1]:
import pandas as pd
import geopandas as gpd

from naturecubepy import (
    get_key,
    get_project,
    auth_headers,
    get_project,
    get_media_assets_df,
    get_project_labels,
    get_station_info,
    get_camera_trap_data,
    get_audio_observation_data,
    get_edna_observation_data
)

In [ ]:
# Retrieve API key and set up authentication headers
api_key = get_key('MY_KEY')
hdr = auth_headers(api_key, "http://127.0.0.1:8000/api/")   

## 1. Get Project Name

In [11]:
project = get_project(hdr)
print(project)

Retrieving project data...
Received response with status code 200
Project data retrieved successfully
Setting your active project as - Germany
boundary=ProjectGeometryResponse(type='FeatureCollection', bbox=None, features=[MultiPolygonGeometryProject(type='Feature', geometry=MultiPolygonModel(type='MultiPolygon', bbox=None, coordinates=[[[Coordinates(lon=13.517485, lat=48.955985, alt=None), Coordinates(lon=13.518071, lat=48.958773, alt=None), Coordinates(lon=13.519936, lat=48.961305, alt=None), Coordinates(lon=13.522899, lat=48.963333, alt=None), Coordinates(lon=13.526668, lat=48.964659, alt=None), Coordinates(lon=13.530876, lat=48.965153, alt=None), Coordinates(lon=13.535109, lat=48.964766, alt=None), Coordinates(lon=13.538954, lat=48.963537, alt=None), Coordinates(lon=13.542033, lat=48.961586, alt=None), Coordinates(lon=13.544046, lat=48.959104, alt=None), Coordinates(lon=13.544796, lat=48.956333, alt=None), Coordinates(lon=13.544208, lat=48.953546, alt=None), Coordinates(lon=13.5423

## 2. Get Stations

Retrieve geolocations and metadata for all measurement stations in your project.

Stations represent the physical or logical deployment sites where data is collected. Each station has:
- **project_system_record_id**: Unique identifier for the station
- **device_id**: QR code or hardware identifier
- **geometry**: GeoJSON point or polygon (longitude, latitude)
- **measurement_type**: `"Camera"`, `"Bioacoustic"`, or `"eDNA"`

The `measurement_type` parameter filters stations by the data they serve:``"camera"``, ``"bioacoustic"``, or ``"eDNA"``.

In [12]:
stations = get_station_info(hdr, measurement_type='all')
print(f"Loaded {len(stations)} stations")
stations.head()

Loaded 18 stations


,geometry,system_type,feature_id,feature_name,system_name,device_id,project_system_record_id,record_count,measurement_type,data_type,project_system_record_start_timestamp,project_system_record_end_timestamp
0,POINT (13.25033 49.1009),Sensor,2513,,Reconyx HP2W HyperFire 2 PROFESSIONAL,104D1WC1,3246,213,Camera,image,2023-11-01T00:00:00+00:00,2024-05-05T22:59:59+00:00
1,POINT (13.23153 49.10139),Sensor,2514,,Reconyx HP2W HyperFire 2 PROFESSIONAL,104D2WC2,3245,61,Camera,image,2024-04-08T23:00:00+00:00,2024-06-05T22:59:59+00:00
2,POINT (13.27662 49.10505),Sensor,2515,,Reconyx HP2W HyperFire 2 PROFESSIONAL,104D3WC3,3244,27,Camera,image,2024-03-20T00:00:00+00:00,2024-06-10T22:59:59+00:00
3,POINT (13.29535 49.08456),Sensor,2516,,Reconyx HP2W HyperFire 2 PROFESSIONAL,104D4WC4,3243,30,Camera,image,2024-04-11T23:00:00+00:00,2024-05-21T22:59:59+00:00
4,POINT (13.4074 48.97583),Sensor,2517,,Reconyx HP2W HyperFire 2 PROFESSIONAL,104D5WC5,3242,34,Camera,image,2024-04-08T23:00:00+00:00,2024-07-09T22:59:59+00:00


In [3]:
from naturecubepy.viz import station_explorer



## 3. Load Label Reference Data

Project labels are the species or taxa used in your study. Can specify where label was detected from (camera or bioacoustic).

In [13]:
for label_type in ["Camera", "Bioacoustic"]:
    print(f"\nProject labels for {label_type}")
    labels = get_project_labels(hdr, label_type, include_iucn_status=True)
    display(labels.head())


Project labels for Camera


,label_id,label,common_name,class_,order,family,genus,species,iucn_redlist_status,tags,global_labels_applied
0,106735,Aves,Bird,Aves,NaN,NaN,NaN,NaN,Least Concern,[],True
1,769863,Bos taurus,Domestic cow,Mammalia,Artiodactyla,Bovidae,Bos,Bos taurus,Not Evaluated,[],True
2,770619,Canis familiaris,Domestic dog,Mammalia,Carnivora,Canidae,Canis,Canis familiaris,Not Evaluated,[],True
3,33236,Canis lupus,Grey Wolf,Mammalia,Carnivora,Canidae,Canis,Canis lupus,Least Concern,[],True
4,28830,Capreolus capreolus,European Roe Deer,Mammalia,Artiodactyla,Cervidae,Capreolus,Capreolus capreolus,Least Concern,[],True



Project labels for Bioacoustic


,label_id,label,common_name,class_,order,family,genus,species,iucn_redlist_status,tags,global_labels_applied
0,49886,Acanthis flammea,Redpoll,Aves,Passeriformes,Fringillidae,Acanthis,Acanthis flammea,Least Concern,[],True
1,51278,Accipiter nisus,Eurasian Sparrowhawk,Aves,Accipitriformes,Accipitridae,Accipiter,Accipiter nisus,Least Concern,[],True
2,41263,Acrocephalus schoenobaenus,Sedge Warbler,Aves,Passeriformes,Acrocephalidae,Acrocephalus,Acrocephalus schoenobaenus,Least Concern,[],True
3,41234,Actitis hypoleucos,Common Sandpiper,Aves,Charadriiformes,Scolopacidae,Actitis,Actitis hypoleucos,Least Concern,[],True
4,41260,Aegithalos caudatus,Long-tailed Tit,Aves,Passeriformes,Aegithalidae,Aegithalos,Aegithalos caudatus,Least Concern,[],True


## 4. Media Assets & Segments

Media are the raw files (images, video, audio, DNA) and their annotations.

- **Media Assets**: Individual files with metadata (duration, size, timestamp, file path)
- **Media Segments**: Logical time windows within a file marked for analysis
  - Each segment may have 0–N labels (species identifications)
  - Includes verification status: `ai_derived`, `labeller_verified`, or `manager_verified`

In [14]:
psr_ids = stations["project_system_record_id"].dropna().astype(int).tolist()

media_assets = get_media_assets_df(hdr, "audio", psr_ids)
print(f"Loaded {len(media_assets)} video media rows from PSR {psr_ids[0]}")
display(media_assets.head())

Loaded 46666 video media rows from PSR 3246


,label_id,label,common_name,class,order,family,genus,species,iucn_redlist_status,tags,...,duration_in_seconds,file_size,number_of_individuals,segment_record_id,label_record_id,prediction_accuracy,manager_verified,labeller_verified,blank,segment_verification_status
0,46271,Certhia familiaris,Eurasian Treecreeper,Aves,Passeriformes,Certhiidae,Certhia,Certhia familiaris,Least Concern,[],...,30.897533,34319813.0,1,719121,1822840,96.684057,False,False,False,ai_derived
1,46271,Certhia familiaris,Eurasian Treecreeper,Aves,Passeriformes,Certhiidae,Certhia,Certhia familiaris,Least Concern,[],...,30.897533,35868841.0,1,545279,1635381,98.735124,False,False,False,ai_derived
2,49897,Fringilla coelebs,Common Chaffinch,Aves,Passeriformes,Fringillidae,Fringilla,Fringilla coelebs,Least Concern,[],...,30.864167,35598738.0,1,719308,1823029,90.375790,False,False,False,ai_derived
3,46271,Certhia familiaris,Eurasian Treecreeper,Aves,Passeriformes,Certhiidae,Certhia,Certhia familiaris,Least Concern,[],...,30.864167,35788979.0,1,545281,1635383,99.387769,False,False,False,ai_derived
4,41216,Caprimulgus europaeus,European Nightjar,Aves,Caprimulgiformes,Caprimulgidae,Caprimulgus,Caprimulgus europaeus,Least Concern,[],...,30.864167,34918349.0,1,545269,1635371,49.292683,False,False,False,ai_derived


## 5. Unified Species Observations

Combine media, labels, and station metadata into flat "observation" records—one row per species identification. Each measurement type (camera, audio, eDNA) calls a different API, but all return consistent columns:

### 5.1 Camera Trap Observations (Image & Video)

`get_camera_trap_data()` returns one row per labelled camera segment, with columns:
- `project_system_record_id`, `device_id`, `latitude`, `longitude`: Station metadata
- `data_type`: `"image"` or `"video"`
- `measurement_type`: `"Camera"`
- `label`, `label_id`, `common_name`, `species`, `genus`, `family`, `order`: Species identification
- `media_file_record_id`, `segment_record_id`, `media_file_created_at`: File/segment IDs and timestamps
- `manager_verified`, `labeller_verified`: Verification status

In [15]:
df = get_camera_trap_data(hdr)
df.head()

,label_id,label,common_name,class,order,family,genus,species,iucn_redlist_status,tags,...,manager_verified,labeller_verified,blank,segment_verification_status,project_system_record_id,device_id,data_type,measurement_type,latitude,longitude
0,770621,Homo sapiens,Human,Mammalia,Primates,Hominidae,Homo,Homo sapiens,Least Concern,[],...,True,False,False,manager_verified,3246,104D1WC1,image,Camera,49.1009,13.25033
1,770621,Homo sapiens,Human,Mammalia,Primates,Hominidae,Homo,Homo sapiens,Least Concern,[],...,True,False,False,manager_verified,3246,104D1WC1,image,Camera,49.1009,13.25033
2,770621,Homo sapiens,Human,Mammalia,Primates,Hominidae,Homo,Homo sapiens,Least Concern,[],...,True,False,False,manager_verified,3246,104D1WC1,image,Camera,49.1009,13.25033
3,770621,Homo sapiens,Human,Mammalia,Primates,Hominidae,Homo,Homo sapiens,Least Concern,[],...,True,False,False,manager_verified,3246,104D1WC1,image,Camera,49.1009,13.25033
4,770621,Homo sapiens,Human,Mammalia,Primates,Hominidae,Homo,Homo sapiens,Least Concern,[],...,True,False,False,manager_verified,3246,104D1WC1,image,Camera,49.1009,13.25033


### 5.2 Audio Observations

`get_audio_observation_data()` returns one row per labelled audio segment, with columns:
- `project_system_record_id`, `device_id`, `latitude`, `longitude`: Station metadata
- `data_type`: `"audio"`
- `measurement_type`: `"Bioacoustic"` `"Camera"`
- `label`, `label_id`, `common_name`, `species`, `genus`, `family`, `order`: Species identification
- `media_file_record_id`, `segment_record_id`, `media_file_created_at`: File/segment IDs and timestamps
- Additional `media_` and `segment_` columns containing full metadata from API responses

In [ ]:
audio = get_audio_observation_data(hdr)
audio.head()

### 5.3 eDNA Observations

`get_edna_observation_data()` returns one row per eDNA environmental DNA asset, with columns:
- `project_system_record_id`, `device_id`, `latitude`, `longitude`: Station metadata
- `data_type`: `"edna"`
- `measurement_type`: `"eDNA"`
- `label`, `label_id`, `common_name`, `species`, `genus`, `family`, `order`: Species identification (from API's species lists)
- `psr_id`: Primary species record ID from eDNA station

In [ ]:
edna = get_edna_observation_data(hdr)
edna.head()